In [1]:
from pathlib import Path

PROJECT = Path(r"C:\Users\LX\Desktop\quebec")
OUTPUT_ROOT = PROJECT / "L_revision"

# Use the confirmed corrected network and shade inputs.
INPUT_MANIFEST = (PROJECT / "revision/fixed_10m_routing_release/07_exact_routing_inputs"
                  / "runs/20260926T235556_632053Z/manifest.json")
EXPECTED_INPUT_MANIFEST_SHA256 = "39046aa90ebb7695786dbae948f8afc4f1827302e7769b7a360dd6cb4063ee83"

ANALYSIS_DATE = "2025-07-15"
HOURS = [11, 12, 13, 14, 15]
SAVE_PHYSICAL_DETAILS = False
SHOW_FIGURES = False

STYLE = {
    "font_family": "Times New Roman",
    "font_scale": 1.5,
    "figsize": (7.55, 6.85),
    "line_width": 2.0,
    "marker_size": 6.0,
    "bar_width": 0.62,
    "y_tick_step": 50,
    "ylim_a": None,
    "ylim_b": None,
    "colors": {"no_shade": "#55585C", "building": "#B96B3A",
               "tree": "#4F876E", "combined": "#315F85"},
    "dpi": 600,
}

In [2]:
import importlib.util

required_packages = ("numpy", "pandas", "matplotlib")
missing_packages = [name for name in required_packages if importlib.util.find_spec(name) is None]
if missing_packages:
    raise ImportError("Missing packages: " + ", ".join(missing_packages))

from datetime import datetime, timezone
from decimal import Decimal, ROUND_HALF_UP
import hashlib
import json
import os
import time
import zipfile

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.ticker import MultipleLocator, FormatStrFormatter

In [3]:
def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def read_csv_checked(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Input file not found: {path}\nCheck the configured project and input paths.')
    frame = pd.read_csv(path, encoding='utf-8-sig')
    require(len(frame) > 0, f'Empty input file: {path}')
    return frame


def numeric(frame, column):
    require(column in frame.columns, f'Missing field: {column}')
    value = pd.to_numeric(frame[column], errors='raise').to_numpy(dtype=float)
    require(np.isfinite(value).all(), f'{column} contains missing or non-finite values.')
    return value


def integer(frame, column):
    value = numeric(frame, column)
    require((np.abs(value) < 2 ** 53).all(), f'{column} exceeds the safe integer range.')
    require(np.allclose(value, np.rint(value), rtol=0, atol=1e-08), f'{column} is not integer-valued.')
    return np.rint(value).astype(np.int64)


def load_radiation(path, analysis_date, hours):
    """Read the user's GEE export in W/m², using its explicit local timestamps."""
    raw = read_csv_checked(path)
    fields = {str(c).strip().lower(): c for c in raw.columns}
    require(len(fields) == len(raw.columns), 'Radiation columns are duplicated after normalization.')
    needed = ['time_local', 'local_hour', 'ssrd_wm2', 'fdir_wm2', 'diffuse_wm2']
    require(all((c in fields for c in needed)), f'Required ERA5 fields: {needed}')
    raw = raw.rename(columns={fields[c]: c for c in needed})
    local = pd.to_datetime(raw.time_local, errors='coerce')
    require(local.notna().all(), 'The ERA5 time_local field contains invalid timestamps.')
    hour_values = integer(raw, 'local_hour')
    require(np.array_equal(hour_values, local.dt.hour.to_numpy()), 'local_hour does not match time_local.')
    chosen = raw.loc[(local.dt.date == pd.Timestamp(analysis_date).date()) & np.isin(hour_values, hours)].copy()
    chosen['hour'] = integer(chosen, 'local_hour')
    chosen = chosen.sort_values('hour').reset_index(drop=True)
    require(chosen.hour.tolist() == list(hours), 'Each analysis hour must have exactly one radiation record.')
    for c in ('ssrd_wm2', 'fdir_wm2', 'diffuse_wm2'):
        chosen[c] = numeric(chosen, c)
        require((chosen[c] >= 0).all(), f'{c} contains negative values.')
    balance = np.abs(chosen.ssrd_wm2 - chosen.fdir_wm2 - chosen.diffuse_wm2)
    require(float(balance.max()) <= 1e-06, 'Total radiation does not equal direct plus diffuse radiation.')
    result = chosen[['hour', 'time_local', 'ssrd_wm2', 'fdir_wm2', 'diffuse_wm2']].copy()
    result['radiation_balance_error_Wm2'] = balance
    return result


def load_fixed_sources(release, source07_run, expected_manifest_sha256, analysis_date, hours):
    release, source07_run = (Path(release), Path(source07_run))
    manifest_path = source07_run / 'manifest.json'
    if not manifest_path.is_file():
        raise FileNotFoundError(f'Prepared input manifest not found: {manifest_path}\nCheck INPUT_MANIFEST and the corrected input release.')
    require(sha256_file(manifest_path) == expected_manifest_sha256, 'The prepared input manifest does not match the confirmed corrected input release.')
    m = json.loads(manifest_path.read_text(encoding='utf-8-sig'))
    require(m['status'] == 'completed' and m['run_id'] == '20260926T235556_632053Z', 'Unexpected prepared-input version or status.')
    require(m['model_version'] == 'fixed10m_min_EDGE_ID_common_distance_v1', 'Unexpected physical-segment input model.')
    require(m['analysis_date'] == analysis_date and m['hours'] == list(hours), 'Prepared-input date or hours do not match the analysis settings.')
    require(m['counts']['physical'] == 53087, 'The confirmed input release must contain 53,087 physical segments.')
    used = [{'role': 'source07_manifest', 'path': str(manifest_path), 'sha256': expected_manifest_sha256}]

    def checked(folder, records, name):
        matches = [r for r in records if r['file'] == name]
        require(len(matches) == 1, f'The input manifest must contain exactly one record for {name}')
        path = folder / name
        if not path.is_file():
            raise FileNotFoundError(f'Corrected input file is missing: {path}')
        digest = sha256_file(path)
        require(digest == matches[0]['sha256'], f'Input file checksum mismatch: {path}')
        used.append({'role': name, 'path': str(path), 'sha256': digest})
        return path
    physical_path = checked(source07_run, m['output_files'], 'physical_segments.csv.gz')
    physical_env_path = checked(source07_run, m['output_files'], 'physical_hour_environment.csv.gz')
    parent_path = checked(source07_run, m['output_files'], 'parent_to_physical.csv.gz')
    parent_env_path = checked(source07_run, m['output_files'], 'parent_hour_environment.csv.gz')
    shade_path = checked(release / 'sources', m['input_files'], 'fixed_10m_physical_shade_hourly.csv.gz')
    radiation_path = checked(release / 'sources', m['input_files'], 'radiation.csv')
    physical = pd.read_csv(physical_path)
    env = pd.read_csv(physical_env_path, dtype={'effective_irradiance_text': str})
    parent = pd.read_csv(parent_path)
    parent_env = pd.read_csv(parent_env_path, dtype={'effective_irradiance_text': str})
    shade = pd.read_csv(shade_path)
    radiation = load_radiation(radiation_path, analysis_date, hours)
    keys = ['physical_segment_id', 'hour']
    require(physical.physical_segment_id.is_unique and len(physical) == 53087, 'Physical-segment identifiers or row count are invalid.')
    require(len(env) == 53087 * 5 and (not env.duplicated(keys).any()), 'Physical-segment hourly inputs are incomplete or duplicated.')
    require(len(shade) == len(env) and (not shade.duplicated(keys).any()), 'Corrected shade-source records are incomplete or duplicated.')
    require(set(env.hour) == set(hours), 'Physical-environment hours are incomplete.')
    require((physical.physical_length_m > 0).all(), 'Physical lengths must be strictly positive.')
    env = env.sort_values(keys).reset_index(drop=True)
    shade = shade.sort_values(keys).reset_index(drop=True)
    require(env[keys].equals(shade[keys]), 'Physical-environment and corrected shade-source keys differ.')
    count_columns = ['n_points', 'n_bldg_shaded', 'n_tree_shaded', 'n_comb_shaded', 'n_both_shaded', 'n_bldg_only', 'n_tree_only', 'n_sunlit']
    for c in ['selected_EDGE_ID'] + count_columns:
        env[c] = integer(env, c)
        require(np.array_equal(env[c], integer(shade, c)), f'Prepared environment and corrected shade source differ for {c}.')
    require(env.selected_parent_edge_id.equals(shade.selected_parent_edge_id), 'The selected sampling parent edge differs between input files.')
    grouped = env.groupby('physical_segment_id', sort=False)
    for c in ['selected_EDGE_ID', 'selected_parent_edge_id', 'n_points', 'physical_length_m']:
        require(grouped[c].nunique().eq(1).all(), f'The five-hour input field {c} is not constant for each physical segment.')
    for h in hours:
        require(set(env.loc[env.hour.eq(h), 'physical_segment_id']) == set(physical.physical_segment_id), f'{h}:00 physical-segment coverage is incomplete.')
    pindex = physical.set_index('physical_segment_id')
    require(np.array_equal(env.selected_EDGE_ID, env.physical_segment_id.map(pindex.source_EDGE_ID)), 'Shade-source EDGE_ID and physical-length source EDGE_ID differ.')
    require(np.allclose(env.physical_length_m, env.physical_segment_id.map(pindex.physical_length_m), rtol=0, atol=1e-09), 'The environment table does not use the common physical lengths.')
    require(parent.parent_edge_id.is_unique and len(parent) == m['counts']['parents'], 'Parent-edge mapping identifiers or row count are invalid.')
    require(len(parent_env) == len(parent) * 5 and (not parent_env.duplicated(['parent_edge_id', 'analysis_hour_local']).any()), 'Directed hourly records are incomplete or duplicated.')
    require(set(parent_env.analysis_hour_local) == set(hours), 'Directed-environment hours are incomplete.')
    pm = parent.set_index('parent_edge_id')
    parent_env['physical_segment_id'] = parent_env.parent_edge_id.map(pm.physical_segment_id)
    require(parent_env.physical_segment_id.notna().all(), 'Some parent edges cannot be mapped to physical segments.')
    require(set(parent.physical_segment_id) == set(physical.physical_segment_id), 'The parent mapping does not cover the physical network.')
    require(np.allclose(parent.parent_length_m, parent.physical_segment_id.map(pindex.physical_length_m), rtol=0, atol=1e-09), 'Opposite parent directions do not use the common physical lengths.')
    joined = parent_env.merge(env[['physical_segment_id', 'hour', 'shade_fraction', 'effective_irradiance_text', 'effective_irradiance_wm2', 'n_points', 'n_comb_shaded']], left_on=['physical_segment_id', 'analysis_hour_local'], right_on=keys, how='left', validate='many_to_one', suffixes=('_parent', '_physical'))
    require(joined.hour.notna().all(), 'Some directed records have no corresponding physical environment.')
    for c in ['effective_irradiance_text', 'shade_fraction', 'n_points', 'n_comb_shaded']:
        require(joined[c + '_parent'].equals(joined[c + '_physical']), f'Opposite parent directions do not share the same physical value for {c}。')
    direction_checks = []
    for h, df in joined.groupby('analysis_hour_local'):
        require(set(df.parent_edge_id) == set(parent.parent_edge_id), f'{h}:00 parent-edge coverage is incomplete.')
        groups = df.groupby('physical_segment_id')
        spreads = groups.shade_fraction_parent.agg(['min', 'max'])
        radiation_spreads = groups.effective_irradiance_wm2_parent.agg(['min', 'max'])
        spread = float((spreads['max'] - spreads['min']).abs().max())
        q_spread = float((radiation_spreads['max'] - radiation_spreads['min']).abs().max())
        require(spread == 0 and q_spread == 0, f'{h}:00 opposite directions still have different shade or irradiance values.')
        direction_checks.append({'hour': int(h), 'physical_segments': len(groups), 'maximum_direction_shade_difference': spread, 'maximum_direction_irradiance_difference_Wm2': q_spread})
    print('Verified prepared input: ', m['run_id'])
    print('Verified 53,087 physical segments across five hours and zero directional differences.')
    return {'env': env, 'radiation': radiation, 'direction_checks': pd.DataFrame(direction_checks), 'source07': m, 'inputs': used}

In [4]:
def percent(numerator, denominator):
    return float(100 * numerator / denominator) if denominator > 0 else np.nan


def physical_hour_for_rq1(env, hour, radiation):
    raw = env.loc[env.hour.eq(hour)].sort_values('physical_segment_id').reset_index(drop=True)
    n = integer(raw, 'n_points')
    require((n > 0).all(), f'{hour}:00 sample-point counts must be strictly positive.')
    counts = {key: integer(raw, col) for key, col in [('building', 'n_bldg_shaded'), ('tree', 'n_tree_shaded'), ('combined', 'n_comb_shaded')]}
    for key, count in counts.items():
        require(((count >= 0) & (count <= n)).all(), f'{hour}:00 {key} shade counts fall outside the valid range.')
    disjoint = {'building_unique': counts['combined'] - counts['tree'], 'tree_unique': counts['combined'] - counts['building'], 'overlap': counts['building'] + counts['tree'] - counts['combined'], 'sunlit': n - counts['combined']}
    require(all(((c >= 0).all() for c in disjoint.values())), f'{hour}:00 shade counts do not satisfy the union identity.')
    for key, col in [('building_unique', 'n_bldg_only'), ('tree_unique', 'n_tree_only'), ('overlap', 'n_both_shaded'), ('sunlit', 'n_sunlit')]:
        require(np.array_equal(disjoint[key], raw[col]), f'{hour}:00 {col} differs from the reconstructed mutually exclusive counts.')
    edge = raw[['physical_segment_id', 'selected_EDGE_ID', 'selected_parent_edge_id']].copy()
    edge['LENGTH_M'] = numeric(raw, 'physical_length_m')
    edge['FREQUENCY'] = n
    qa = {'hour': hour, 'physical_segments': len(raw), 'sample_records': int(n.sum())}
    for key, code in [('building', 'bldg'), ('tree', 'tree'), ('combined', 'comb')]:
        edge[key + '_shade_share'] = counts[key] / n
        error = float(np.max(np.abs(edge[key + '_shade_share'] - numeric(raw, f'shade_{code}_frac'))))
        require(error <= 1e-14, f'{hour}:00 {key} fraction differs from the corrected point counts.')
        qa[key + '_mean_error'] = error
    for key, count in disjoint.items():
        edge[key + '_share'] = count / n
    require(np.allclose(raw.direct_wm2, radiation['fdir_wm2'], rtol=0, atol=1e-09) and np.allclose(raw.diffuse_wm2, radiation['diffuse_wm2'], rtol=0, atol=1e-09), 'Radiation differs from the prepared input release.')
    q = float(radiation['diffuse_wm2']) + float(radiation['fdir_wm2']) * (1 - edge.combined_shade_share)
    qa['effective_irradiance_vs07_error_Wm2'] = float(np.max(np.abs(q - numeric(raw, 'effective_irradiance_wm2'))))
    require(qa['effective_irradiance_vs07_error_Wm2'] <= 1e-09, 'RQ1 irradiance does not match the prepared routing environment.')
    return (edge, qa)


def compute_hour(edge, radiation, analysis_date):
    """All network means are length weighted over physical segments, each counted once."""
    hour = int(radiation['hour'])
    length = edge.LENGTH_M.to_numpy(dtype=float)
    network_length = float(length.sum())
    total = float(radiation['ssrd_wm2'])
    direct = float(radiation['fdir_wm2'])
    diffuse = float(radiation['diffuse_wm2'])
    b = edge.building_shade_share.to_numpy()
    t = edge.tree_shade_share.to_numpy()
    u = edge.combined_shade_share.to_numpy()
    overlap = edge.overlap_share.to_numpy()
    exposure = {'no_shade_exposure_Wm2': np.full(len(edge), total), 'building_only_exposure_Wm2': diffuse + direct * (1 - b), 'tree_only_exposure_Wm2': diffuse + direct * (1 - t), 'observed_exposure_Wm2': diffuse + direct * (1 - u)}
    reduction = {'total_attenuation_Wm2': direct * u, 'building_unique_attenuation_Wm2': direct * edge.building_unique_share.to_numpy(), 'tree_unique_attenuation_Wm2': direct * edge.tree_unique_share.to_numpy(), 'overlap_attenuation_Wm2': direct * overlap, 'building_shapley_attenuation_Wm2': direct * (edge.building_unique_share.to_numpy() + 0.5 * overlap), 'tree_shapley_attenuation_Wm2': direct * (edge.tree_unique_share.to_numpy() + 0.5 * overlap)}
    e0, eb, et, ebt = (exposure[k] for k in exposure)
    phi_b = reduction['building_shapley_attenuation_Wm2']
    phi_t = reduction['tree_shapley_attenuation_Wm2']
    qa = {'exposure_balance_error_Wm2': float(np.max(np.abs(e0 - ebt - reduction['total_attenuation_Wm2']))), 'shapley_balance_error_Wm2': float(np.max(np.abs(phi_b + phi_t - reduction['total_attenuation_Wm2']))), 'building_shapley_formula_error_Wm2': float(np.max(np.abs(phi_b - 0.5 * (e0 - eb + (et - ebt))))), 'tree_shapley_formula_error_Wm2': float(np.max(np.abs(phi_t - 0.5 * (e0 - et + (eb - ebt))))), 'scenario_order_error_Wm2': float(max(0, np.max(ebt - eb), np.max(ebt - et), np.max(eb - e0), np.max(et - e0)))}
    require(max(qa.values()) <= 1e-06, f'{hour}:00 exposure or attribution checks failed: {qa}')
    weighted = lambda a: float(np.dot(length, a) / network_length)
    summary = {'analysis_date': analysis_date, 'hour': hour, 'physical_segments': len(edge), 'physical_network_length_m': network_length, 'total_shortwave_Wm2': total, 'direct_Wm2': direct, 'diffuse_Wm2': diffuse}
    for source in ('building', 'tree', 'combined'):
        summary[source + '_shade_pct'] = 100 * weighted(edge[source + '_shade_share'])
    for source in ('building_unique', 'tree_unique', 'overlap', 'sunlit'):
        summary[source + '_shade_pct'] = 100 * weighted(edge[source + '_share'])
    for key, values in {**exposure, **reduction}.items():
        summary[key] = weighted(values)
        summary[key.replace('_Wm2', '_network_cost_W_per_m')] = float(np.dot(length, values))
    attenuation = summary['total_attenuation_Wm2']
    summary['total_attenuation_pct'] = percent(attenuation, summary['no_shade_exposure_Wm2'])
    summary['building_shapley_share_pct'] = percent(summary['building_shapley_attenuation_Wm2'], attenuation)
    summary['tree_shapley_share_pct'] = percent(summary['tree_shapley_attenuation_Wm2'], attenuation)
    details = edge.copy()
    details.insert(0, 'hour', hour)
    details.insert(0, 'analysis_date', analysis_date)
    for key, values in {**exposure, **reduction}.items():
        details[key] = values
    return (summary, qa, details)


def five_hour_summary(summary):
    """Equal-hour means, not a time integral. Attribution shares use summed reductions."""
    columns = [c for c in summary.columns if c.endswith('_Wm2') or c.endswith('_shade_pct')]
    row = {'analysis_date': summary.analysis_date.iloc[0], 'period': '11:00-15:00', 'analysis_hours': len(summary), 'physical_segments': int(summary.physical_segments.iloc[0]), 'physical_network_length_m': float(summary.physical_network_length_m.iloc[0])}
    row.update({c: float(summary[c].mean()) for c in columns})
    row['total_attenuation_pct'] = percent(summary.total_attenuation_Wm2.sum(), summary.no_shade_exposure_Wm2.sum())
    row['building_shapley_share_pct'] = percent(summary.building_shapley_attenuation_Wm2.sum(), summary.total_attenuation_Wm2.sum())
    row['tree_shapley_share_pct'] = percent(summary.tree_shapley_attenuation_Wm2.sum(), summary.total_attenuation_Wm2.sum())
    return pd.DataFrame([row])

In [5]:
TABLE_COLUMNS = ['total_shortwave_Wm2', 'direct_Wm2', 'building_shade_pct', 'tree_shade_pct', 'overlap_shade_pct', 'combined_shade_pct', 'observed_exposure_Wm2', 'total_attenuation_Wm2', 'total_attenuation_pct', 'building_shapley_share_pct', 'tree_shapley_share_pct']

TABLE_HEADERS = ['Total shortwave (W/m²)', 'Direct (W/m²)', 'Building shade (%)', 'Tree shade (%)', 'Overlap (%)', 'Combined shade (%)', 'Mean effective irradiance (W/m²)', 'Total reduction (W/m²)', 'Reduction share (%)', 'Building contribution (%)', 'Tree contribution (%)']

FIGURE_COLUMNS = ['no_shade_exposure_Wm2', 'building_only_exposure_Wm2', 'tree_only_exposure_Wm2', 'observed_exposure_Wm2', 'total_attenuation_Wm2', 'building_shapley_attenuation_Wm2', 'tree_shapley_attenuation_Wm2', 'building_shapley_share_pct', 'tree_shapley_share_pct']

TABLE_CAPTION = 'Hourly radiation, shade composition, and reductions in mean effective irradiance across the pedestrian network.'

TABLE_NOTE = 'Radiation values are hourly means. Shade shares and mean effective irradiance under existing shade are weighted by physical segment length. Building and tree shade overlap and are therefore not additive, while combined shade represents their union. Mean effective irradiance is calculated with both building and tree shade included, and total reduction is measured relative to no shade. Contribution shares partition this reduction between buildings and trees, with the reduction from overlapping shade split equally between them.'

FIGURE_CAPTION = 'Counterfactual mean effective irradiance and attribution of shade-induced reductions across the pedestrian network. Panel (a) compares length-weighted mean effective irradiance under no shade, building shade only, tree shade only, and the observed combination of building and tree shade. Panel (b) partitions the total reduction between building and tree shade while accounting for overlapping shade.'


def fmt2(value):
    """Round only final display values to two decimal places and retain trailing zeros."""
    return format(Decimal(str(value)).quantize(Decimal('0.01'), rounding=ROUND_HALF_UP), '.2f')


def formatted_table(hourly):
    result = pd.DataFrame({'Time': [f'{int(h):02d}:00' for h in hourly.hour]})
    for col, heading in zip(TABLE_COLUMNS, TABLE_HEADERS):
        result[heading] = [fmt2(value) for value in hourly[col]]
    return result


def table_latex(hourly):
    prefix = '\\begin{table*}[!htbp]\n\\centering\n\\caption{%s}\n\\label{tab:rq1_environmental_formation}\n\\small\n\\setlength{\\tabcolsep}{2.8pt}\n\\renewcommand{\\arraystretch}{1.0}\n\\begin{tabular*}{\\textwidth}{@{\\extracolsep{\\fill}}l*{11}{r}@{}}\n\\toprule\n& \\multicolumn{2}{c}{\\shortstack{Radiation\\\\($\\mathrm{W\\,m^{-2}}$)}}\n& \\multicolumn{4}{c}{\\shortstack{Length-weighted shade\\\\(\\%%)}}\n& \\multicolumn{3}{c}{\\shortstack{Effective irradiance\\\\and reduction}}\n& \\multicolumn{2}{c}{\\shortstack{Attributed contribution\\\\(\\%%)}} \\\\\n\\cmidrule(lr){2-3}\\cmidrule(lr){4-7}\\cmidrule(lr){8-10}\\cmidrule(l){11-12}\nTime\n& \\shortstack{Total\\\\shortwave} & Direct & Building & Tree & Overlap & Combined\n& \\shortstack{Mean effective\\\\irradiance\\\\($\\mathrm{W\\,m^{-2}}$)}\n& \\shortstack{Total\\\\reduction\\\\($\\mathrm{W\\,m^{-2}}$)}\n& \\shortstack{Reduction\\\\share\\\\(\\%%)} & Building & Tree \\\\\n\\midrule\n' % TABLE_CAPTION
    rows = []
    for _, row in hourly.iterrows():
        rows.append(' & '.join([f"{int(row['hour']):02d}:00"] + [fmt2(row[c]) for c in TABLE_COLUMNS]) + ' \\\\')
    suffix = '\n\\bottomrule\n\\end{tabular*}\n\\vspace{3pt}\n\\begin{minipage}{\\textwidth}\n\\footnotesize\n\\textit{Note:} %s\n\\end{minipage}\n\\end{table*}\n' % TABLE_NOTE
    return prefix + '\n'.join(rows) + suffix


def figure_latex():
    return '\\begin{figure}[htbp]\n    \\centering\n    \\includegraphics[width=0.75\\textwidth]{figures/rq1_shade_attribution.pdf}\n    \\caption{%s}\n    \\label{fig:shade_attribution}\n\\end{figure}\n' % FIGURE_CAPTION

In [6]:
def choose_font(preferred):
    for path in Path('/usr/share/fonts/opentype/urw-base35').glob('NimbusRoman-*.otf'):
        font_manager.fontManager.addfont(str(path))
    available = {font.name for font in font_manager.fontManager.ttflist}
    return next((f for f in [preferred, 'Nimbus Roman', 'Liberation Serif', 'DejaVu Serif'] if f in available), 'DejaVu Serif')


def plot_shade_attribution(hourly, output, style):
    font = choose_font(style['font_family'])
    scale, colors = (style['font_scale'], style['colors'])
    h = hourly.hour.astype(int).to_numpy()
    d = hourly[FIGURE_COLUMNS].astype(float)
    step = float(style['y_tick_step'])
    require(step > 0, 'y_tick_step must be positive.')
    with mpl.rc_context({'font.family': font, 'mathtext.fontset': 'stix', 'font.size': 8.5 * scale, 'axes.labelsize': 8.5 * scale, 'xtick.labelsize': 8 * scale, 'ytick.labelsize': 8 * scale, 'legend.fontsize': 7.5 * scale, 'axes.linewidth': 0.8, 'axes.labelpad': 7, 'pdf.fonttype': 42, 'ps.fonttype': 42, 'figure.facecolor': 'white', 'savefig.facecolor': 'white'}):
        fig, (a, b) = plt.subplots(2, 1, figsize=style['figsize'], sharex=True, layout='constrained')
        for col, label, color, linestyle, marker in [('no_shade_exposure_Wm2', 'No shade', 'no_shade', '--', 'o'), ('building_only_exposure_Wm2', 'Building shade only', 'building', '-.', 's'), ('tree_only_exposure_Wm2', 'Tree shade only', 'tree', ':', '^'), ('observed_exposure_Wm2', 'Observed building + tree', 'combined', '-', 'D')]:
            a.plot(h, d[col], label=label, color=colors[color], linestyle=linestyle, marker=marker, linewidth=style['line_width'], markersize=style['marker_size'], markeredgecolor='white', markeredgewidth=0.6, zorder=3)
        values = d[FIGURE_COLUMNS[:4]].to_numpy()
        lower = max(0, step * np.floor((values.min() - 40) / step))
        upper = step * np.ceil((values.max() + 150) / step)
        a.set_ylim(style['ylim_a'] if style['ylim_a'] is not None else (lower, upper))
        a.set_ylabel('Mean effective irradiance\n' + '(W m$^{-2}$)')
        a.legend(loc='upper center', ncol=2, frameon=False, handlelength=2, columnspacing=1.2, borderaxespad=0.65)
        a.tick_params(axis='x', bottom=False, labelbottom=False)
        building = d.building_shapley_attenuation_Wm2.to_numpy()
        tree = d.tree_shapley_attenuation_Wm2.to_numpy()
        total = d.total_attenuation_Wm2.to_numpy()
        b.bar(h, building, width=style['bar_width'], color=colors['building'], edgecolor='white', linewidth=0.8, label='Building contribution', zorder=3)
        b.bar(h, tree, bottom=building, width=style['bar_width'], color=colors['tree'], edgecolor='white', linewidth=0.8, label='Tree contribution', zorder=3)
        upper = max(step, step * np.ceil((total.max() * 1.3 + 20) / step))
        b.set_ylim(style['ylim_b'] if style['ylim_b'] is not None else (0, upper))
        b.set_ylabel('Reduction in mean\neffective irradiance\n' + '(W m$^{-2}$)')
        b.set_xlabel('Time of day')
        b.legend(loc='upper center', ncol=2, frameon=False, handlelength=1.9, columnspacing=1.6, borderaxespad=0.65)
        for i, hour in enumerate(h):
            row = hourly.iloc[i]
            for y, key in [(building[i] / 2, 'building_shapley_share_pct'), (building[i] + tree[i] / 2, 'tree_shapley_share_pct')]:
                b.text(hour, y, fmt2(row[key]) + '%', ha='center', va='center', color='white', fontsize=8 * scale, fontweight='bold', zorder=4)
            b.annotate(fmt2(row['total_attenuation_Wm2']), (hour, total[i]), xytext=(0, 5), textcoords='offset points', ha='center', va='bottom', fontsize=7.5 * scale)
        for ax, tag in [(a, '(a)'), (b, '(b)')]:
            ax.spines[['top', 'right']].set_visible(False)
            ax.set_axisbelow(True)
            ax.grid(axis='y', color='#DDDCD8', linewidth=0.65, alpha=0.8)
            ax.set_xlim(h.min() - 0.55, h.max() + 0.55)
            ax.yaxis.set_major_locator(MultipleLocator(step))
            ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
            ax.text(0.008, 0.98, tag, transform=ax.transAxes, ha='left', va='top', fontsize=9 * scale, fontweight='bold')
        b.set_xticks(h, [f'{x}:00' for x in h])
        paths = []
        for extension in ['pdf', 'png']:
            path = output / f'rq1_shade_attribution.{extension}'
            fig.savefig(path, dpi=style['dpi'], bbox_inches='tight', pad_inches=0.06)
            paths.append(path)
        plt.close(fig)
    return (paths, font)

In [7]:
def atomic_json(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(temporary, path)


def run_rq1(input_manifest, expected_input_hash, analysis_date, hours, output_root,
            style, save_physical_details=False, show_figures=False):
    started = time.perf_counter()
    input_manifest = Path(input_manifest)
    source07_run = input_manifest.parent
    release = source07_run.parents[2]
    stage = Path(output_root) / "rq1_shade_contributions"
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    output = stage / "runs" / run_id
    output.mkdir(parents=True, exist_ok=False)
    tables, figures = output / "tables", output / "figures"
    tables.mkdir()
    figures.mkdir()
    status = {
        "analysis": "rq1_shade_contributions", "status": "running", "run_id": run_id,
        "analysis_date": analysis_date, "hours": list(hours),
        "analysis_unit": "physical segment, each counted once",
        "network_weight": "common physical_length_m",
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "input_manifest": str(input_manifest), "input_manifest_sha256": expected_input_hash,
        "display_decimal_places": 2,
    }
    atomic_json(output / "manifest.json", status)
    try:
        sources = load_fixed_sources(release, source07_run, expected_input_hash, analysis_date, hours)
        atomic_json(output / "input_files.json", sources["inputs"])
        rows, checks = [], []
        for hour in hours:
            radiation = sources["radiation"].loc[sources["radiation"].hour.eq(hour)].iloc[0]
            physical, check = physical_hour_for_rq1(sources["env"], hour, radiation)
            summary, calculation_checks, details = compute_hour(physical, radiation, analysis_date)
            rows.append(summary)
            check.update(calculation_checks)
            checks.append(check)
            if save_physical_details:
                folder = output / "physical_details"
                folder.mkdir(exist_ok=True)
                details.to_csv(folder / f"rq1_physical_hour_{hour}.csv.gz", index=False, compression="gzip")
            print(f"{hour}:00 complete: {len(physical):,} physical segments; "
                  f"mean effective irradiance {summary['observed_exposure_Wm2']:.2f} W/m²")

        hourly = pd.DataFrame(rows)
        means = five_hour_summary(hourly)
        raw_outputs = {
            "rq1_hourly_full.csv": hourly,
            "rq1_five_hour_mean_full.csv": means,
            "rq1_figure_data.csv": hourly[["hour"] + FIGURE_COLUMNS],
            "rq1_input_and_calculation_checks.csv": pd.DataFrame(checks),
            "rq1_direction_consistency_checks.csv": sources["direction_checks"],
            "rq1_radiation_selected.csv": sources["radiation"],
        }
        # Preserve full precision in machine-readable results; round only publication outputs.
        for name, frame in raw_outputs.items():
            frame.to_csv(tables / name, index=False, encoding="utf-8-sig", float_format="%.17g")
        publication_table = formatted_table(hourly)
        publication_table.to_csv(tables / "rq1_hourly_table.csv", index=False, encoding="utf-8-sig")
        (tables / "rq1_hourly_table.tex").write_text(table_latex(hourly), encoding="utf-8")
        (figures / "rq1_shade_attribution.tex").write_text(figure_latex(), encoding="utf-8")
        mean_columns = ["total_shortwave_Wm2", "observed_exposure_Wm2", "total_attenuation_Wm2",
                        "total_attenuation_pct", "building_shapley_attenuation_Wm2",
                        "tree_shapley_attenuation_Wm2", "building_shapley_share_pct", "tree_shapley_share_pct"]
        pd.DataFrame({"Metric": mean_columns, "Value": [fmt2(means[c].iloc[0]) for c in mean_columns]}).to_csv(
            tables / "rq1_five_hour_summary.csv", index=False, encoding="utf-8-sig")
        figure_files, font = plot_shade_attribution(hourly, figures, style)

        status.update(
            status="completed", source07_run_id=sources["source07"]["run_id"],
            model_version=sources["source07"]["model_version"],
            physical_segments=int(hourly.physical_segments.iloc[0]),
            physical_hour_rows=len(sources["env"]),
            physical_network_length_m=float(hourly.physical_network_length_m.iloc[0]),
            all_checks_passed=True, font_used=font,
            wall_seconds=time.perf_counter()-started,
            finished_utc=datetime.now(timezone.utc).isoformat(),
        )
        status["output_files"] = [
            {"file": p.relative_to(output).as_posix(), "bytes": p.stat().st_size, "sha256": sha256_file(p)}
            for p in sorted(output.rglob("*")) if p.is_file() and p.name != "manifest.json"
        ]
        atomic_json(output / "manifest.json", status)
        bundle = output / "rq1_shade_contributions.zip"
        with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
            for path in sorted(output.rglob("*")):
                if path.is_file() and path != bundle:
                    archive.write(path, path.relative_to(output).as_posix())
        atomic_json(stage / "LATEST_SUCCESS.json", {
            "run_id": run_id, "manifest": str(output / "manifest.json"),
            "manifest_sha256": sha256_file(output / "manifest.json"),
        })
        print(publication_table.to_string(index=False))
        print(f"Output: {output}")
        print(f"Results archive: {bundle}")
        if show_figures:
            from IPython.display import display, Image
            display(Image(filename=str(figures / "rq1_shade_attribution.png"), width=850))
        return output
    except (Exception, KeyboardInterrupt) as error:
        status.update(status="interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
                      error_type=type(error).__name__, error=str(error),
                      finished_utc=datetime.now(timezone.utc).isoformat())
        atomic_json(output / "manifest.json", status)
        raise

In [8]:
RQ1_RUN = run_rq1(
    INPUT_MANIFEST, EXPECTED_INPUT_MANIFEST_SHA256, ANALYSIS_DATE, HOURS,
    OUTPUT_ROOT, STYLE, save_physical_details=SAVE_PHYSICAL_DETAILS,
    show_figures=SHOW_FIGURES,
)

Verified prepared input:  20260926T235556_632053Z
Verified 53,087 physical segments across five hours and zero directional differences.
11:00 complete: 53,087 physical segments; mean effective irradiance 382.21 W/m²
12:00 complete: 53,087 physical segments; mean effective irradiance 503.01 W/m²
13:00 complete: 53,087 physical segments; mean effective irradiance 424.39 W/m²
14:00 complete: 53,087 physical segments; mean effective irradiance 353.70 W/m²
15:00 complete: 53,087 physical segments; mean effective irradiance 361.28 W/m²
 Time Total shortwave (W/m²) Direct (W/m²) Building shade (%) Tree shade (%) Overlap (%) Combined shade (%) Mean effective irradiance (W/m²) Total reduction (W/m²) Reduction share (%) Building contribution (%) Tree contribution (%)
11:00                 669.54        519.15              22.53          49.31       16.49              55.34                           382.21                 287.33               42.91                     25.81                 74.19
